In [1]:
import pandas as pd
import  numpy as np
pd.set_option("display.max_columns",None)

In [3]:
train_dataset = pd.read_csv(
    "../data/processed/sepsis_progression_train.csv"
)

print("Training dataset shape:", train_dataset.shape)
print("\nColumns:")
print(train_dataset.columns.tolist())

Training dataset shape: (300569, 129)

Columns:
['hour', 'heart_rate', 'resp_rate', 'temperature', 'sbp', 'dbp', 'mbp', 'spo2', 'gcs', 'creatinine', 'bun', 'urineoutput_last', 'urineoutput_sum', 'urineoutput_24hr', 'wbc', 'hemoglobin', 'hematocrit', 'platelet', 'bands', 'sodium', 'potassium', 'chloride', 'bicarbonate', 'calcium', 'magnesium', 'aniongap', 'albumin', 'bilirubin_total', 'bilirubin_max', 'inr', 'pt', 'ptt', 'crp', 'lactate', 'pao2fio2ratio_novent', 'pao2fio2ratio_vent', 'glucose_lab', 'shock_index', 'map_calculated', 'bun_creatinine_ratio', 'spo2_deficit', 'heart_rate_prev', 'heart_rate_delta', 'resp_rate_prev', 'resp_rate_delta', 'temperature_prev', 'temperature_delta', 'sbp_prev', 'sbp_delta', 'mbp_prev', 'mbp_delta', 'spo2_prev', 'spo2_delta', 'gcs_prev', 'gcs_delta', 'heart_rate_roll3_mean', 'heart_rate_roll6_mean', 'sofa_score_x', 'sofa_score_y', 'sofa_score', 'sofa_prev', 'sofa_delta_1h', 'sofa_roll3_mean', 'sofa_roll3_max', 'sofa_roll3_min', 'sofa_roll6_mean', 'sofa

In [4]:
print("Potential ID/time columns:")

for column in train_dataset.columns:
    if any(keyword in column.lower()
           for keyword in ["stay", "subject", "patient", "hour", "time"]):
        print(column)

Potential ID/time columns:
hour
creatinine_hours_since
bun_hours_since
wbc_hours_since
hemoglobin_hours_since
hematocrit_hours_since
platelet_hours_since
bands_hours_since
sodium_hours_since
potassium_hours_since
chloride_hours_since
bicarbonate_hours_since
calcium_hours_since
magnesium_hours_since
aniongap_hours_since
albumin_hours_since
bilirubin_total_hours_since
bilirubin_max_hours_since
inr_hours_since
pt_hours_since
ptt_hours_since
crp_hours_since
lactate_hours_since
glucose_lab_hours_since
pao2fio2ratio_novent_hours_since
pao2fio2ratio_vent_hours_since


In [5]:
target_column = "progression_class"
class_names = {
    0:"Improving",
    1:"Stable",
    2:"Deteriorating"
}
print(" number of historical snapshots:", len(train_dataset))
print("\nprogression class distribution:")
class_counts = train_dataset[target_column].value_counts().sort_index()
for class_index, count in class_counts.items():
    percentage = count / len(train_dataset) * 100
    print(
        f"{class_index} = {class_names[class_index]:15s} "
        f"{count:8d} ({percentage:.2f}%)"
    )
    print("\nmissing values:")
    print(
        train_dataset.isna()
        .sum()
        .sort_values(ascending=False)
        .head(15)
    )

 number of historical snapshots: 300569

progression class distribution:
0 = Improving          85039 (28.29%)

missing values:
crp                     300326
bands                   298750
pao2fio2ratio_novent    297596
crp_hours_since         296422
crp_ffill               296422
albumin                 295957
bilirubin_max           291407
bilirubin_total         291401
liver                   291365
gcs_delta               283778
inr                     282679
pt                      282674
ptt                     282256
pao2fio2ratio_vent      278294
lactate                 278106
dtype: int64
1 = Stable            146483 (48.74%)

missing values:
crp                     300326
bands                   298750
pao2fio2ratio_novent    297596
crp_hours_since         296422
crp_ffill               296422
albumin                 295957
bilirubin_max           291407
bilirubin_total         291401
liver                   291365
gcs_delta               283778
inr                     28267

In [6]:
### check candidate's case rag feature
## we can't use all the feature's from dataset so we r going to use clinical state and trajectory for the library

In [8]:
candidate_feature = [
    "heart_rate",
    "resp_rate",
    "temperature",
    "sbp",
    "dbp",
    "mbp",
    "spo2",
    "gcs",
    "creatinine_ffill",
    "bun_ffill",
    "wbc_ffill",
    "lactate_ffill",
    "sofa_score_x",
    "sofa_prev",
    "sofa_delta_1h",
    "sofa_roll3_mean",
    "sofa_roll6_mean",
    "shock_index",
    "spo2_deficit"
]
available_feature = [
    feature for feature in  candidate_feature
    if feature in train_dataset.columns
]
missing_feature = [
    feature for feature in candidate_feature
    if feature not in available_feature
]
print("available candidate features:")
print(available_feature)
print("\n features not found:")
print(missing_feature)

available candidate features:
['heart_rate', 'resp_rate', 'temperature', 'sbp', 'dbp', 'mbp', 'spo2', 'gcs', 'creatinine_ffill', 'bun_ffill', 'wbc_ffill', 'lactate_ffill', 'sofa_score_x', 'sofa_prev', 'sofa_delta_1h', 'sofa_roll3_mean', 'sofa_roll6_mean', 'shock_index', 'spo2_deficit']

 features not found:
[]


### building case library

In [9]:
case_feature = [
     "heart_rate",
    "resp_rate",
    "temperature",
    "sbp",
    "dbp",
    "mbp",
    "spo2",
    "gcs",
    "creatinine_ffill",
    "bun_ffill",
    "wbc_ffill",
    "lactate_ffill",
    "sofa_score_x",
    "sofa_prev",
    "sofa_delta_1h",
    "sofa_roll3_mean",
    "sofa_roll6_mean",
    "shock_index",
    "spo2_deficit"
]
case_library = train_dataset[
    case_feature + ["progression_class"]
].copy()
### readable outcome
case_library["progression"] = (
    case_library["progression_class"].map(class_names)
)
print(" case library shape:", case_library.shape)
print("\n case library columns:")
print(case_library.columns.tolist())
print("\n case library 5 cases:")
print(case_library.head(5))

 case library shape: (300569, 21)

 case library columns:
['heart_rate', 'resp_rate', 'temperature', 'sbp', 'dbp', 'mbp', 'spo2', 'gcs', 'creatinine_ffill', 'bun_ffill', 'wbc_ffill', 'lactate_ffill', 'sofa_score_x', 'sofa_prev', 'sofa_delta_1h', 'sofa_roll3_mean', 'sofa_roll6_mean', 'shock_index', 'spo2_deficit', 'progression_class', 'progression']

 case library 5 cases:
   heart_rate  resp_rate  temperature    sbp   dbp   mbp   spo2  gcs  \
0        88.0       14.0          NaN   92.0  43.0  55.0  100.0  NaN   
1        87.0       13.0          NaN   97.0  51.0  62.0  100.0  NaN   
2        92.0       18.0          NaN  128.0  66.0  82.0   98.0  NaN   
3        83.0       14.0          NaN   78.0  42.0  50.0  100.0  NaN   
4        81.0       15.0        35.56   86.0  42.0  52.0  100.0  NaN   

   creatinine_ffill  bun_ffill  wbc_ffill  lactate_ffill  sofa_score_x  \
0               NaN        NaN        NaN            NaN           1.0   
1               NaN        NaN        NaN   